In [1]:
from collections import Counter

import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

def word_frequency(text):
  words = word_tokenize(text.lower())

  return Counter(words)

text = "NLP is amazing, NLP makes machines understand language"

print(word_frequency(text))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Counter({'nlp': 2, 'is': 1, 'amazing': 1, ',': 1, 'makes': 1, 'machines': 1, 'understand': 1, 'language': 1})


In [2]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "NLP is fun and amazing",
    "Machine understand NLP and Text",
    "Text processing is a part of NLP"
]

vectorizer = CountVectorizer()
X =vectorizer.fit_transform(corpus)
len(vectorizer.get_feature_names_out())

11

In [3]:
X.toarray()

array([[1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1],
       [0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0]])

In [4]:
# TFIDF

from sklearn.feature_extraction.text import TfidfVectorizer
vector = TfidfVectorizer()
X = vector.fit_transform(corpus)

vector.get_feature_names_out()

array(['amazing', 'and', 'fun', 'is', 'machine', 'nlp', 'of', 'part',
       'processing', 'text', 'understand'], dtype=object)

In [5]:
X.toarray()

array([[0.53409337, 0.40619178, 0.53409337, 0.40619178, 0.        ,
        0.31544415, 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 0.40619178, 0.        , 0.        , 0.53409337,
        0.31544415, 0.        , 0.        , 0.        , 0.40619178,
        0.53409337],
       [0.        , 0.        , 0.        , 0.35829137, 0.        ,
        0.27824521, 0.4711101 , 0.4711101 , 0.4711101 , 0.35829137,
        0.        ]])

In [6]:
import numpy as np
import math

tokenized_docs = [doc.lower().split() for doc in corpus]
tokenized_docs

[['nlp', 'is', 'fun', 'and', 'amazing'],
 ['machine', 'understand', 'nlp', 'and', 'text'],
 ['text', 'processing', 'is', 'a', 'part', 'of', 'nlp']]

In [7]:
vocab = sorted(set(word for doc in tokenized_docs for word in doc))
vocab_index = {word: idx for idx, word in enumerate(vocab)}
print(vocab)
print(vocab_index)

['a', 'amazing', 'and', 'fun', 'is', 'machine', 'nlp', 'of', 'part', 'processing', 'text', 'understand']
{'a': 0, 'amazing': 1, 'and': 2, 'fun': 3, 'is': 4, 'machine': 5, 'nlp': 6, 'of': 7, 'part': 8, 'processing': 9, 'text': 10, 'understand': 11}


In [8]:
def compute_tf(doc_tokens):
  tf_vector = np.zeros(len(vocab))
  words_count = Counter(doc_tokens)
  for word, count in words_count.items():
    tf_vector[vocab_index[word]] = count/len(doc_tokens)
  return tf_vector

tf_matrix = np.array([compute_tf(doc) for doc in tokenized_docs])
# print(tf_matrix)


def compute_idf():
  N =len(tokenized_docs)
  idf= np.zeros(len(vocab))
  for word, idx in vocab_index.items():
    df = sum(1 for doc in tokenized_docs if word in doc)
    idf[idx] =math.log((N+1)/ (df+1) +1)
  return idf

idf_vector = compute_idf()
print(idf_vector)

[1.09861229 1.09861229 0.84729786 1.09861229 0.84729786 1.09861229
 0.69314718 1.09861229 1.09861229 1.09861229 0.84729786 1.09861229]


In [9]:
tfidf_matrix = tf_matrix * idf_vector
print(tfidf_matrix)

[[0.         0.21972246 0.16945957 0.21972246 0.16945957 0.
  0.13862944 0.         0.         0.         0.         0.        ]
 [0.         0.         0.16945957 0.         0.         0.21972246
  0.13862944 0.         0.         0.         0.16945957 0.21972246]
 [0.15694461 0.         0.         0.         0.12104255 0.
  0.09902103 0.15694461 0.15694461 0.15694461 0.12104255 0.        ]]


In [12]:
import numpy as np
import pandas as pd
df = pd.DataFrame(tfidf_matrix, columns=vocab)
df.round(3)

,a,amazing,and,fun,is,machine,nlp,of,part,processing,text,understand
0,0.000,0.22,0.169,0.22,0.169,0.00,0.139,0.000,0.000,0.000,0.000,0.00
1,0.000,0.00,0.169,0.00,0.000,0.22,0.139,0.000,0.000,0.000,0.169,0.22
2,0.157,0.00,0.000,0.00,0.121,0.00,0.099,0.157,0.157,0.157,0.121,0.00


In [ ]:
#NLP Task -Feature Extraction

#NLP Task -Feature Extraction

In [13]:
import numpy as np

feature_array = np.array(vectorizer.get_feature_names_out())
important = np.argsort(X.toarray()).flatten()[::-1]

keywords= feature_array[important[:5]]
print(keywords)

['processing' 'of' 'part' 'is' 'text']


In [15]:
top_n=2
for i, row in enumerate(tfidf_matrix):
  top_indices = row.argsort()[-top_n:][::-1]
  keywords = [vocab[idx] for idx in top_indices]
  print(keywords)

['fun', 'amazing']
['understand', 'machine']
['part', 'processing']
